# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [487]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt_tab", quiet=True) # Need "punkt_tab" for compile
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [488]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)
# implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# apply sent_tokenize

# Searching points between letters and replacing with symbols for more easy replacement later
protect_acronym = re.sub(r'(?<=[A-Z])\.', '###', text)
print(protect_acronym)

# Apply sent_tokenize without posible errors
tokenize = sent_tokenize(protect_acronym)
print(tokenize)

# Replacing ### on the tokenized sentence with [] for no get <generator object <genexpr> at 0x0000028383ABF840>
sentences = [i.replace("###", ".") for i in tokenize]
print(sentences)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U###P###C### and U###N###E###S###C###O### A report valued the project at $3.2 billion.
['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U###P###C### and U###N###E###S###C###O### A report valued the project at $3.2 billion.']
['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.']


## Q2

In [489]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm

# Remove dots of acronyms
remove_acronym_dots = re.sub(r'(?<=[A-Z])\.', '', text)
print(remove_acronym_dots)

# Meters to centimeters with lambda for the calculations
convert_m_cm = re.sub(r'(\d+\.\d+)m', 
lambda x: str(int(float(x.group(1))*100)) + " centimeters", remove_acronym_dots)
print(convert_m_cm)

#
number_text = convert_m_cm.replace("$3.2 billion", "three point two billion")
print(number_text)

text_norm = number_text
print(text_norm)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from UPC and UNESCO A report valued the project at $3.2 billion.
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at $3.2 billion.
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [490]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

# Pattern that links words beginning with a capital letter followed by lowercase letters with an underscore "_" to later "dodge" them
pattern_lowercase = re.sub(r'([A-Z][a-z]+)\s+([A-Z][a-z]+)', r'\1_\2', text_norm)
print(pattern_lowercase)

# Declaration of variables that we will use later
words = pattern_lowercase.split() # Everything in a list to change word by word
final_result = []
for i in words:
    if i.isupper() or "_" in i or i == "OpenAI": # Words all upper or with "_" are excluded, and OpenAI is a mixed one
        final_result.append(i)
    else:
        final_result.append(i.lower())


text_case = " ".join(final_result) # Join the created list to change the words with a " " between them
text_case = text_case.replace("_"," ") # Removing _ from Sam_Altman
print(text_case)


In mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.
in mid-february 2026, the CEO of openai, Sam Altman, visited barcelona. he is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q4

In [491]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = word_tokenize (text)

print(tokens)


['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '1.86m', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'U.P.C', '.', 'and', 'U.N.E.S.C.O', '.', 'A', 'report', 'valued', 'the', 'project', 'at', '$', '3.2', 'billion', '.']


## Q5

In [492]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

# Declare the stop words
stopwords = set(stopwords.words('english'))

# Iterate through the tokenized list to remove the stopwords
tokens_nostop = [t for t in tokens if t not in stopwords]

print(tokens_nostop)


['In', 'mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', '1.86m', 'tall', 'met', 'researchers', 'U.P.C', '.', 'U.N.E.S.C.O', '.', 'A', 'report', 'valued', 'project', '$', '3.2', 'billion', '.']


## Q6

In [493]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]
bigrams = []

for i in range(len(tokens_nostop) - 1): # In the list range -1 because it takes two at a time, and with a range of its own it takes 1 less, therefore -2
    bigrams.append( (tokens_nostop[i], tokens_nostop[i+1]) ) # Add words two at a time, using i and i+1, to the bigram list


print(bigrams)


[('In', 'mid-February'), ('mid-February', '2026'), ('2026', ','), (',', 'CEO'), ('CEO', 'OpenAI'), ('OpenAI', ','), (',', 'Sam'), ('Sam', 'Altman'), ('Altman', ','), (',', 'visited'), ('visited', 'Barcelona'), ('Barcelona', '.'), ('.', 'He'), ('He', '1.86m'), ('1.86m', 'tall'), ('tall', 'met'), ('met', 'researchers'), ('researchers', 'U.P.C'), ('U.P.C', '.'), ('.', 'U.N.E.S.C.O'), ('U.N.E.S.C.O', '.'), ('.', 'A'), ('A', 'report'), ('report', 'valued'), ('valued', 'project'), ('project', '$'), ('$', '3.2'), ('3.2', 'billion'), ('billion', '.')]


## Q7

In [ ]:
from collections import defaultdict # Necesary for the model
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = defaultdict(int)
model = None
counts = defaultdict(lambda: defaultdict(int))

for i in range(len(bigrams)-1):
        w1, w2 = bigrams[i], bigrams[i+1]
        bigrams_count[w1][w2] +=1
        context_counts[w1] += 1

def predict_next(prev_word, model, top_k=3):
    return None

# Example:
print(predict_next("OpenAI", model, top_k=3))


SyntaxError: 'return' outside function (2094625518.py, line 25)

## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [ ]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
